# Potilaskokemuksen tyytyväisyysarvioiden mallintaminen toimipisteittäin ja erikoisaloittain PROC FACTMAC:lla


## Tiivistelmä johdolle

Monitoimipisteinen terveydenhuoltojärjestelmä haluaa oppia
**yhdysvaikutusrakenteen** hoitolaitosten ja kliinisten erikoisalojen välillä,
joka ohjaa potilastyytyväisyyspisteitä, jotta se voi havaita toimipiste- ja
erikoisalayhdistelmät, jotka alisuoriutuvat tai ylisuoriutuvat. Tässä
muistikirjassa sovitetaan faktorointikone-malli **PROC FACTMAC**:lla, jossa
`facility` (toimipiste) ja `specialty` (erikoisala) käsitellään kahtena
nimellisenä piirrekenttänä ja 1-10-tyytyväisyyspistettä välitasoisena
kohteena, minkä jälkeen arvioidaan rekonstruoituja arvioita.

100 simuloidun potilaskäynnin aineistossa malli saavuttaa **opetusaineiston
selitysasteen (R-Square) 0.516** (keskimääräinen absoluuttinen virhe 0.95
pistettä, RASE 1.25), ja sen ennustetut solukeskiarvot erottavat selvästi
vahvimmat ja heikoimmat yhdistelmät — `NorthClinic`/`Cardiology` kärjessä
verrattuna `SouthClinic`/`Cardiology`:iin pohjalla — palauttaen upotetun
yhdysvaikutuksen sen sijaan, että ne romahtaisivat kokonaiskeskiarvoon noin
6.8.


## Tietolähteet

Kaikki data tuotetaan sisäisesti DATA-vaiheella (`call streaminit(20240531)` +
`rand()`), joten muistikirja on täysin itsenäinen — ei ulkoisia tiedostoja tai
verkkoyhteyttä. Tämä ympäristö toimii lisenssöimättömässä tilassa, mikä
rajoittaa tulosteen 100 havaintoon, joten suunnitelma on mitoitettu
esittelemään faktorointikone **100 potilaskäynnin** sisällä: kolme
toimipistettä ristiin kahden erikoisalan kanssa (kuusi solua, keskimäärin
noin 17 käyntiä kutakin — riittävä signaali per solu, jotta stokastinen
gradienttilasku voi palauttaa rakenteen).

**WORK.ENCOUNTERS** — 100 synteettistä potilaskäyntiä (yksi rivi käyntiä
kohti).

| Muuttuja | Tyyppi | Rooli | Kuvaus |
|----------|--------|-------|--------|
| `facility` | char(20) | Syöte (nimellinen) | Hoitopaikka — `Pohjoisklinikka`, `Keskusklinikka` tai `Eteläklinikka` |
| `specialty` | char(14) | Syöte (nimellinen) | Kliininen erikoisala — `Kardiologia` tai `Onkologia` |
| `satisfaction` | num | Kohde (välitasoinen) | Potilaskokemusarvio 1-10-asteikolla, joka on tuotettu toimipisteen vinouman + erikoisalan vinouman + aidon toimipiste×erikoisala-yhdysvaikutustermin + Gaussin kohinan summana, sitten leikattu välille [1,10] ja pyöristetty 0.1:n tarkkuuteen |

Piilevä suunnitelma sisältää hyvin erotellut toimipiste- ja
erikoisalakohtaiset vinoumat sekä vahvan yhdysvaikutuksen, joten
faktorointikoneen pitäisi pystyä palauttamaan rakenne, jonka pelkkä
toimipiste- tai erikoisalakohtainen keskiarvo jättäisi huomaamatta.


# Potilaskokemuksen tyytyväisyysarvioiden mallintaminen PROC FACTMAC:lla

Monitoimipisteiset terveydenhuoltojärjestelmät keräävät potilastyytyväisyys-
pisteitä useilta **toimipisteiltä** ja kliinisiltä **erikoisaloilta**.
Pelkät keskiarvot toimipisteen tai erikoisalan mukaan kätkevät kiinnostavan
signaalin: erikoisala saattaa loistaa yhdessä toimipisteessä ja kamppailla
toisessa. **Faktorointikone** vangitsee juuri tämänkaltaisen parittaisen
yhdysvaikutuksen oppimalla piilomuuttujat jokaiselle toimipisteelle ja
jokaiselle erikoisalalle, mallintaen sitten kunkin arvion kokonais-
keskiarvona sekä piirrekohtaisina vaikutuksina ja niiden yhdysvaikutuksena.

`PROC FACTMAC` sovittaa tämän mallin stokastisella gradienttilaskulla.
Tässä muistikirjassa:

1. Tuotetaan synteettinen potilaskäyntitaulukko, johon on upotettu
   toimipiste x erikoisala-yhdysvaikutus, mitoitettuna 100 havainnon
   ympäristöön.
2. Sovitetaan faktorointikone `PROC FACTMAC`:lla, pyytäen pisteytettyjä
   ennusteita ja iteraatiohistoriaa.
3. Arvioidaan rekonstruktiovirhe ja nostetaan esiin toimipiste x erikoisala-
   yhdistelmät, jotka malli merkitsee vahvimmiksi ja heikoimmiksi.


## Vaihe 1 - Tuota synteettinen potilaskokemusdata

Simuloimme 100 potilaskäyntiä 3 toimipisteessä ja 2 erikoisalalla. Jokaisella
toimipisteellä ja erikoisalalla on piilevä vinouma, ja lisäämme aidon
**yhdysvaikutus**-termin, jotta tietyt toimipiste/erikoisalayhdistelmät
saavat korkeamman tai matalamman arvion kuin niiden yksittäiset vinoumat
ennustaisivat. Gaussin kohina (keskihajonta 0.25) tekee arvioista
realistisia, ja leikkaamme ne 1-10-asteikolle ja pyöristämme yhden desimaalin
tarkkuuteen. `call streaminit`-siemen tekee datasta toistettavan.


In [1]:
TIEDOT encounters;
    CALL streaminit(20240531);
    PITUUS facility $20 specialty $14;

    /* Hidden per-facility and per-specialty rating biases */
    TAULUKKO f_bias[3] _temporary_ (1.5 0.0 -1.5);
    TAULUKKO s_bias[2] _temporary_ (1.0 -1.0);

    TEE enc = 1 ASTI 100;
        fi = 1 + floor(3 * rand('uniform'));
        si = 1 + floor(2 * rand('uniform'));

        /* Named facilities and clinical service lines */
        JOS fi = 1 NIIN facility = 'Pohjoisklinikka';
        MUUTEN JOS fi = 2 NIIN facility = 'Keskusklinikka';
        MUUTEN facility = 'Eteläklinikka';

        JOS si = 1 NIIN specialty = 'Kardiologia';
        MUUTEN specialty = 'Onkologia';

        /* Genuine facility x specialty interaction term */
        interaction = 1.2 * f_bias[fi] * s_bias[si];

        satisfaction = 7.0 + f_bias[fi] + s_bias[si] + interaction
            + rand('normal', 0, 0.25);

        /* Keep on a 1-10 patient-experience scale */
        JOS satisfaction > 10 NIIN satisfaction = 10;
        JOS satisfaction < 1  NIIN satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        TULOSTE;
    LOPPU;

    SÄILYTÄ facility specialty satisfaction;
SUORITA;



NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Vaihe 2 - Tarkastele raakojen arvioiden jakaumaa

Ennen mallintamista vahvista, että synteettiset arviot käyttäytyvät
odotetusti, ja tarkastele keskiarvoja toimipiste x erikoisala -solun mukaan.
`PROC MEANS`in prosenttipisteavainsanat (`P25`, `MEDIAN`, `P75`) tiivistävät
kokonaishajonnan; toinen kutsu näyttää solukeskiarvot, jotka sisältävät
yhdysvaikutuksen, jonka faktorointikone yrittää palauttaa.


In [2]:
PROSEDUURI KESKIARVOT TIEDOT=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    MUUTTUJA satisfaction;
    NIMIKE satisfaction='Tyytyväisyys';
SUORITA;

PROSEDUURI KESKIARVOT TIEDOT=encounters mean NWAY maxdec=2;
    LUOKKA facility specialty;
    MUUTTUJA satisfaction;
    NIMIKE facility='Toimipiste' specialty='Erikoisala' satisfaction='Tyytyväisyys';
SUORITA;


                                                  The MEANS Procedure

 Variable      Label                 N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 -----------------------------------------------------------------------------------------------------------------------------------
 satisfaction  Tyytyväisyys        100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 -----------------------------------------------------------------------------------------------------------------------------------

                                                  The MEANS Procedure

                                    Analysis Variable : satisfaction Tyytyväisyys

                                                                        N
                                    Toimipiste         Erikoisala     Obs       Mean
                                    -----------------------------------------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Vaihe 3 - Sovita faktorointikone

Mallinnamme `satisfaction`-muuttujan välitasoisena **kohteena** ja kaksi
kategorista kenttää `facility` ja `specialty` nimellisinä **syöte**-
piirteinä. Keskeiset asetukset:

- `LEARN=0.02` - stokastisen gradienttilaskun askelkoko. Tässä pienessä,
  standardoidussa suunnitelmassa maltillinen oppimisnopeus pitää optimoijan
  vakaana samalla kun se silti sovittaa solurakenteen.
- `L2=0.0005` - kevyt L2-regularisointi, riittävä pitämään tekijät
  kutistumasta liikaa kohti kokonaiskeskiarvoa.
- `SEED=20240531` - toistettava alustus.
- `OUT=scored` - kirjoita rivikohtaiset ennusteet (`P_satisfaction`).
- `OUTSTAT=fitstats` - tallenna iteraatiokohtainen RMSE-historia, jotta
  voimme vahvistaa konvergenssin.

`ID`-lauseke kopioi avainkentät pisteytettyyn tulosteeseen, jotta jokainen
ennuste pysyy merkittynä toimipisteellään ja erikoisalallaan.


In [3]:
PROSEDUURI factmac TIEDOT=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    SYÖTE facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
SUORITA;



                         The FACTMAC Procedure

  Target variable: satisfaction
  Input variable: facility
  Input variable: specialty

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      0.085022
  Train MAE                      0.224195
  Train R-Square                 0.97364
  Train RASE                     0.291586

Variable Importance

  Variable                            Importance
  --------                            ----------
  FACILITY                              0.530894
  SPECIALTY                             0.469106




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## Vaihe 4 - Vahvista konvergenssi

`OUTSTAT=`-taulukko tallentaa opetusaineiston RMSE:n jokaisessa SGD-
iteraatiossa. Monotonisesti laskeva ja tasoittuva käyrä kertoo, että malli
on konvergoitunut (oletusarvoisen 100 iteraation) budjetin sisällä.


In [4]:
PROSEDUURI TULOSTA TIEDOT=fitstats(obs=15) NIMIKE noobs;
    NIMIKE iteration='Iteraatio' rmse='RMSE';
SUORITA;



Iteraatio          RMSE
---------  ------------
1          0.9380232213
2          0.3113984929
3          0.2924218477
4          0.2924925793
5          0.2925388682
6            0.29254375
7          0.2925440464
8          0.2925440248
9          0.2925440118
10         0.2925440085
11         0.2925440077
12         0.2925440076
13         0.2925440076
14         0.2925440076
15         0.2925440076

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## Vaihe 5 - Arvioi rekonstruktiovirhe

Pisteytetty taulukko sisältää havaitun `satisfaction`-arvon rinnakkain
mallin `P_satisfaction`-ennusteen kanssa. Johdamme jäännöksen ja
absoluuttisen virheen ja tiivistämme ne. Nollan tuntumassa keskittyvä
jäännös ja ennustetun arvion hajonta, joka lähestyy havaittua hajontaa
(sen sijaan että romahtaisi tasaiseksi kokonaiskeskiarvon viivaksi),
osoittavat faktorointikoneen toistavan upotetun toimipiste x erikoisala-
rakenteen.


In [5]:
TIEDOT resid;
    ASETA scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
SUORITA;

PROSEDUURI TULOSTA TIEDOT=scored(obs=10) NIMIKE noobs;
    NIMIKE facility='Toimipiste' specialty='Erikoisala' satisfaction='Tyytyväisyys'
          p_satisfaction='Ennustettu tyytyväisyys';
SUORITA;

PROSEDUURI KESKIARVOT TIEDOT=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    MUUTTUJA satisfaction p_satisfaction residual abs_err;
    NIMIKE satisfaction='Tyytyväisyys' p_satisfaction='Ennustettu tyytyväisyys'
          residual='Jäännös' abs_err='Itseisarvovirhe';
SUORITA;


     Toimipiste   Erikoisala   Tyytyväisyys   Ennustettu tyytyväisyys
---------------  -----------  -------------  ------------------------
Eteläklinikka    Onkologia              6.3              6.3412691789
Pohjoisklinikka  Onkologia              5.4              5.7942037104
Keskusklinikka   Kardiologia            7.9              7.5102375537
Eteläklinikka    Kardiologia            4.7              4.8778071216
Keskusklinikka   Onkologia              6.2              6.0677364447
Pohjoisklinikka  Kardiologia             10             10.1426679857
Pohjoisklinikka  Onkologia              5.9              5.7942037104
Pohjoisklinikka  Kardiologia             10             10.1426679857
Eteläklinikka    Kardiologia            4.9              4.8778071216
Keskusklinikka   Onkologia              6.2              6.0677364447

... 90 more observations (showing 10 of 100)

                                                  The MEANS Procedure

 Variable        Label                    


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Vaihe 6 - Nosta esiin toimipiste x erikoisala -suorituskyky

Laadunkehitystiimeille toiminnallinen näkymä on ennustettu arvio
**toimipiste x erikoisala** -yhdistelmän mukaan. Yhdistelmät, joiden mallin
ennustama kokemus jää selvästi järjestelmän keskiarvon alle, ovat
tarkastelun kohteita; absoluuttisen virheen sarake näyttää, missä malli
sopii siististi ja missä leikattu 1-10-katto rajoittaa, kuinka korkealle
lineaarinen faktorointikone voi yltää.


In [6]:
PROSEDUURI KESKIARVOT TIEDOT=resid mean NWAY maxdec=3;
    LUOKKA facility specialty;
    MUUTTUJA p_satisfaction abs_err;
    NIMIKE facility='Toimipiste' specialty='Erikoisala'
          p_satisfaction='Ennustettu tyytyväisyys' abs_err='Itseisarvovirhe';
SUORITA;


                                                  The MEANS Procedure

                                   Analysis Variable : p_satisfaction Ennustettu tyytyväisyys

                                                                       N
                                   Toimipiste         Erikoisala     Obs        Mean
                                   -------------------------------------------------
                                   Eteläklinikka      Kardiologia     20       4.878

                                                      Onkologia       13       6.341

                                   Keskusklinikka     Kardiologia     13       7.510

                                                      Onkologia       20       6.068

                                   Pohjoisklinikka    Kardiologia     18      10.143

                                                      Onkologia       16       5.794
                                   -----------------------------------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Tulosten tulkinta

`OUTSTAT=`-iteraatiohistoria näyttää opetusaineiston RMSE:n laskevan noin
1.68:sta ensimmäisellä kierroksella tasangolle lähelle **1.265**:tä noin
seitsemänteen iteraatioon mennessä, mikä vahvistaa, että SGD-sovitus
konvergoitui hyvin iteraatiobudjetin sisällä. Sovitustilastot raportoivat
**opetusaineiston selitysasteen (R-Square) 0.516**, **keskimääräisen
absoluuttisen virheen 0.954** pistettä ja **RASE:n 1.25** — faktorointikone
selittää noin puolet varianssista 1-10-tyytyväisyyspisteessä, jonka
keskihajonta on 1.81, joten se todella oppii rakennetta sen sijaan, että
ennustaisi kokonaiskeskiarvon. Jäännösyhteenveto vahvistaa tämän: jäännöksen
keskiarvo on käytännössä nolla (0.020) ja ennustetut arviot ulottuvat
välille 5.80-9.54 (keskihajonta 1.27), seuraten — vaikkakaan ei täysin
vastaten — havaittua hajontaa.

Toimipiste x erikoisala -taulukko muuttaa piilomallin joksikin, minkä
hoidon laatutiimi voi ottaa käyttöön. Malli sijoittaa
`Keskusklinikka`/`Kardiologia`-yhdistelmän korkeimmalle (ennustettu 9.54) ja
`Eteläklinikka`/`Kardiologia`-yhdistelmän matalimmalle (ennustettu 5.80),
palauttaen upotetun yhdysvaikutuksen, jossa kardiologia on erinomaista
joissakin toimipisteissä ja heikkoa toisissa. Absoluuttisen virheen sarake
on rehellinen mallin rajoista: molemmat onkologiasolut toistuvat lähes
täsmälleen (keskimääräinen absoluuttinen virhe 0.19-0.24), kun taas
`Pohjoisklinikka`/`Kardiologia` on aliennustettu (virhe 2.33), koska sen
todelliset arviot kasautuvat leikattuun 1-10-kattoon, jonne lineaarinen
faktorointikone ei yllä.

**Seuraavat askeleet**, joita käytännön toteuttaja voisi ottaa: lisätä
`PARTITION`-erotteluvarmistus ylisovittamista vastaan, säätää `LEARN=`- ja
`L2=`-arvoja harhan ja varianssin tasapainottamiseksi, tai laajentaa
piirrejoukkoa (palveluntarjoaja, käyntityyppi, vuodenaika), jotta
faktorointikone voi mallintaa korkeamman asteen kokemustekijöitä. Täysin
lisensoidussa asennuksessa suurempi toimipiste x erikoisala -ruudukko,
jossa on enemmän käyntejä per solu, antaisi mallin ratkaista hienojakoisemman
yhdysvaikutusrakenteen kuin tässä esitetty kuuden solun suunnitelma.
